In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install mne matplotlib numpy pandas

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

from mne.io import read_raw_edf, concatenate_raws
from mne.channels import make_standard_montage
from mne.preprocessing import ICA

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, FastICA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

In [ ]:
# Create a list of data file paths
background_files = [
    '/content/drive/MyDrive/hw2eeg/Subject19_1.edf',
    '/content/drive/MyDrive/hw2eeg/Subject20_1.edf'
]
task_files = [
    '/content/drive/MyDrive/hw2eeg/Subject19_2.edf',
    '/content/drive/MyDrive/hw2eeg/Subject20_2.edf'
]

# Load Raw objects for each condition
raws_background = [mne.io.read_raw_edf(f, preload=True) for f in background_files]
raws_task = [mne.io.read_raw_edf(f, preload=True) for f in task_files]

# Concatenate data from both subjects
raw_background = mne.io.concatenate_raws(raws_background)
raw_task = mne.io.concatenate_raws(raws_task)

# Rename channels and set montage for each Raw object
for raw in [raw_background, raw_task]:
    rename_dict = {
        'EEG Fp1': 'Fp1', 'EEG Fp2': 'Fp2', 'EEG F3': 'F3', 'EEG F4': 'F4',
        'EEG F7': 'F7', 'EEG F8': 'F8', 'EEG T3': 'T3', 'EEG T4': 'T4',
        'EEG C3': 'C3', 'EEG C4': 'C4', 'EEG T5': 'T5', 'EEG T6': 'T6',
        'EEG P3': 'P3', 'EEG P4': 'P4', 'EEG O1': 'O1', 'EEG O2': 'O2',
        'EEG Fz': 'Fz', 'EEG Cz': 'Cz', 'EEG Pz': 'Pz',
        'EEG A2-A1': 'A2',
        'ECG ECG': 'ECG'
    }
    raw.rename_channels(rename_dict)
    raw.set_channel_types({'ECG': 'ecg'})
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))

print("Data loading and preprocessing from Google Drive complete.")
print("\nInfo for the background condition data:")
print(raw_background.info)
print("\nInfo for the task condition data:")
print(raw_task.info)

Extracting EDF parameters from /content/drive/MyDrive/hw2eeg/Subject19_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Extracting EDF parameters from /content/drive/MyDrive/hw2eeg/Subject20_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Extracting EDF parameters from /content/drive/MyDrive/hw2eeg/Subject19_2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.000 ...    61.998 secs...
Extracting EDF parameters from /content/drive/MyDrive/hw2eeg/Subject20_2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.000 ...    61.998 secs...
Data loading and preprocessing from Google Drive complete.

Info for the background condition data:
<Info | 9 non-empty valu

In [ ]:
# Extract X, Y data

# Select only EEG channels (exclude ECG channel)
raw_background.pick('eeg')
raw_task.pick('eeg')

# Get data using .get_data() and transpose (.T) to (samples, channels) shape
X_background = raw_background.get_data().T
X_task = raw_task.get_data().T

# Create labels (0 = background, 1 = task)
Y_background = np.zeros(X_background.shape[0], dtype=int) # 0 for background
Y_task = np.ones(X_task.shape[0], dtype=int)     # 1 for task

# Concatenate data: Combine X and Y data into single large arrays
X_all = np.concatenate([X_background, X_task])
Y_all = np.concatenate([Y_background, Y_task])

print(f"X_all shape (samples, channels): {X_all.shape}")
print(f"Y_all shape (samples,): {Y_all.shape}")
print(f"Total samples: {len(Y_all)}, Task samples: {np.sum(Y_all == 1)}, Background samples: {np.sum(Y_all == 0)}")

X_all shape (samples, channels): (244000, 20)
Y_all shape (samples,): (244000,)
Total samples: 244000, Task samples: 62000, Background samples: 182000


## Question (a) 50/50 Split + Logistic Regression

In [ ]:
# 1. Split data 50/50 using slicing
X_train = X_all[0::2, :] # Use even-indexed samples for training
Y_train = Y_all[0::2]
X_test = X_all[1::2, :]  # Use odd-indexed samples for testing
Y_test = Y_all[1::2]

print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# 2. Standardize data
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

# 3. Train Logistic Regression model
clf_lr_a = LogisticRegression(max_iter=1000, random_state=42)
clf_lr_a.fit(X_train_norm, Y_train)

# 4. Evaluate the model
preds_a = clf_lr_a.predict(X_test_norm)

# Calculate all three required metrics
acc_a = accuracy_score(Y_test, preds_a)
bal_acc_a = balanced_accuracy_score(Y_test, preds_a)
f1_a = f1_score(Y_test, preds_a)

# 5. Report results and provide interpretation
print("\nPart (a) Results")
print(f"50/50 Split Accuracy: {acc_a:.4f}")
print(f"50/50 Split Balanced Accuracy: {bal_acc_a:.4f}")
print(f"50/50 Split F1 Score: {f1_a:.4f}")

Train set shape: (122000, 20), Test set shape: (122000, 20)

Part (a) Results
50/50 Split Accuracy: 0.7459
50/50 Split Balanced Accuracy: 0.5000
50/50 Split F1 Score: 0.0000


## Interpretation for Part (a)
- Accuracy: Percentage of correct predictions (Total Correct / Total Samples)-> 0.7459 (74.6%)

- Balanced Accuracy: Average accuracy per class. Important if one class has more samples -> 0.5000 (50.0%)

- F1 Score: Harmonic mean of Precision and Recall. Also good for imbalanced data -> 0.0000 (0.0%)



The model has failed. The high Accuracy (74.6%) is misleading because the dataset is imbalanced, with Class 0 (Background) making up 74.6% of the data. The model simply learned to predict the majority class (Class 0) for every sample.

This failure is confirmed by the other metrics:

The Balanced Accuracy of 50.0% indicates performance is no better than random guessing .

The F1 Score of 0.0% shows the model completely failed to identify any Class 1 (Task) samples .


### Conclusion:
The model could not find a meaningful pattern in the noisy, high-dimensional raw EEG data  and defaulted to guessing the majority class. This demonstrates that Accuracy is a poor metric here and that Feature Engineering is essential.

## Question (b) 5-Fold CV + Logistic Regression

In [ ]:
# 1. Setup 5-Fold CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler_cv = StandardScaler()
clf_lr_cv = LogisticRegression(max_iter=1000, random_state=42)

# Lists to store scores from each of the 5 folds
scores_acc_b = []
scores_bal_acc_b = []
scores_f1_b = []

# 2. CV Loop
print("Starting 5-Fold Cross-Validation...")
for i, (train_idx, test_idx) in enumerate(kf.split(X_all, Y_all)):
    # Split data for this fold
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]

    # Standardize data
    X_train_norm = scaler_cv.fit_transform(X_train)
    X_test_norm = scaler_cv.transform(X_test)

    # Fit and predict
    clf_lr_cv.fit(X_train_norm, Y_train)
    preds_b = clf_lr_cv.predict(X_test_norm)

    # Calculate metrics for this fold
    acc_fold = accuracy_score(Y_test, preds_b)
    bal_acc_fold = balanced_accuracy_score(Y_test, preds_b)
    f1_fold = f1_score(Y_test, preds_b)

    # Print the result for fold
    print(f"--- Fold {i+1}/5 Results ---")
    print(f"  Accuracy: {acc_fold:.4f}")
    print(f"  Balanced Accuracy: {bal_acc_fold:.4f}")
    print(f"  F1 Score: {f1_fold:.4f}")

    # Store scores
    scores_acc_b.append(acc_fold)
    scores_bal_acc_b.append(bal_acc_fold)
    scores_f1_b.append(f1_fold)

# 3. Report average results
print("\n--- Part (b) Final Average Results ---")
print(f"5-Fold CV Average Accuracy: {np.mean(scores_acc_b):.4f} (+/- {np.std(scores_acc_b):.4f})")
print(f"5-Fold CV Average Balanced Accuracy: {np.mean(scores_bal_acc_b):.4f}")
print(f"5-Fold CV Average F1 Score: {np.mean(scores_f1_b):.4f}")

Starting 5-Fold Cross-Validation...
--- Fold 1/5 Results ---
  Accuracy: 0.7459
  Balanced Accuracy: 0.5000
  F1 Score: 0.0000
--- Fold 2/5 Results ---
  Accuracy: 0.7459
  Balanced Accuracy: 0.5000
  F1 Score: 0.0000
--- Fold 3/5 Results ---
  Accuracy: 0.7459
  Balanced Accuracy: 0.5000
  F1 Score: 0.0000
--- Fold 4/5 Results ---
  Accuracy: 0.7459
  Balanced Accuracy: 0.5000
  F1 Score: 0.0000
--- Fold 5/5 Results ---
  Accuracy: 0.7459
  Balanced Accuracy: 0.5000
  F1 Score: 0.0000

--- Part (b) Final Average Results ---
5-Fold CV Average Accuracy: 0.7459 (+/- 0.0000)
5-Fold CV Average Balanced Accuracy: 0.5000
5-Fold CV Average F1 Score: 0.0000


## Interpretation for Part (b)
**Comparison:**
The average results from the 5-fold Cross-Validation (CV) in Part (b) are **identical** to the results from the single 50/50 split in Part (a).
* **Average Accuracy:** 0.7459 (with 0.0000 std)
* **Average Balanced Accuracy:** 0.5000
* **Average F1 Score:** 0.0000

The 5-fold CV confirms that the model's failure in Part (a) was not a random chance due to a "bad" split. The model is consistently failing in the exact same way across all 5 folds.

The primary cause for this identical result is the **severe class imbalance** in the dataset (74.6% Class 0 vs. 25.4% Class 1).Because the model is trained on **noisy, high-dimensional raw EEG data** , it cannot find a meaningful pattern to distinguish the classes.

Therefore, the model defaults to the simplest strategy in every fold: **predicting the majority class (Class 0) for every sample**. This strategy consistently yields:
1.  An **Accuracy** of 74.6% (matching the majority class percentage).
2.  A **Balanced Accuracy** of 50.0% (the average of 100% correct for Class 0 and 0% correct for Class 1), which is equivalent to random guessing.
3.  An **F1 Score** of 0.0%, as it never identifies any Class 1 samples.

Cross-validation is an *evaluation* method; it does not fix the underlying data problem. In this case, it reliably confirmed that the raw data is unsuitable for this model, motivating the need for **Feature Engineering** in Part (c).

## Question (c) Feature Engineering + 5-Fold CV

In [ ]:
# Feature Method 1: PCA
print("Running CV on Feature Set 1 (PCA)...")

kf_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler_c1 = StandardScaler()
clf_lr_c1 = LogisticRegression(max_iter=1000, random_state=42)
pca = PCA(n_components=10) # Reduce 19 channels to 10 components

scores_acc_c1 = []
for train_idx, test_idx in kf_c1.split(X_all, Y_all): # Use X_all, Y_all
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]

    X_train_norm = scaler_c1.fit_transform(X_train)
    X_test_norm = scaler_c1.transform(X_test)

    # Apply PCA (L17, Slide 12)
    X_train_pca = pca.fit_transform(X_train_norm)
    X_test_pca = pca.transform(X_test_norm)

    clf_lr_c1.fit(X_train_pca, Y_train)
    scores_acc_c1.append(accuracy_score(Y_test, clf_lr_c1.predict(X_test_pca)))

Running CV on Feature Set 1 (PCA)...


In [ ]:
#Feature Method 2: ICA
print("Running CV on Feature Set 2 (ICA)...")

kf_c2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler_c2 = StandardScaler()
clf_lr_c2 = LogisticRegression(max_iter=1000, random_state=42)
ica = FastICA(n_components=10, random_state=42, max_iter=1000) # Reduce to 10 components

scores_acc_c2 = []
for train_idx, test_idx in kf_c2.split(X_all, Y_all): # Use X_all, Y_all
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]

    X_train_norm = scaler_c2.fit_transform(X_train)
    X_test_norm = scaler_c2.transform(X_test)

    # Apply ICA
    X_train_ica = ica.fit_transform(X_train_norm)
    X_test_ica = ica.transform(X_test_norm)

    clf_lr_c2.fit(X_train_ica, Y_train)
    scores_acc_c2.append(accuracy_score(Y_test, clf_lr_c2.predict(X_test_ica)))

Running CV on Feature Set 2 (ICA)...


In [ ]:
# Feature Method 3: PSD Band Power
print("Creating Epochs for PSD features...")

#the original MNE objects
raw_all = concatenate_raws([raw_background, raw_task])

# Create 2-second, non-overlapping epochs
epochs = mne.make_fixed_length_epochs(raw_all, duration=2.0, preload=True, verbose=False)

epoch_midpoints = (epochs.events[:, 0] - raw_all.first_samp + epochs.tmax * raw_all.info['sfreq']).astype(int)
Y_feat3 = Y_all[epoch_midpoints]

# Compute PSD
psd_data = epochs.compute_psd(method='welch', fmin=1., fmax=50., verbose=False).get_data()
freqs = epochs.compute_psd(verbose=False).freqs

# Define brainwave bands
bands = {'Alpha': (8., 13.), 'Beta': (13., 30.)}
X_psd_features = []

# Calculate average power in each band
for fmin, fmax in bands.values():
    freq_idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    # Get mean power across frequency band (axis=2)
    band_power = np.mean(psd_data[:, :, freq_idx], axis=2) # Shape: (n_epochs, n_channels)
    X_psd_features.append(band_power)

# Concatenate Alpha and Beta power features
X_feat3 = np.concatenate(X_psd_features, axis=1) # Shape: (n_epochs, n_channels * 2)

print(f"PSD Feature shape: {X_feat3.shape}")
print("Running CV on Feature Set 3 (PSD)...")
kf_c3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler_c3 = StandardScaler()
clf_lr_c3 = LogisticRegression(max_iter=1000, random_state=42)

scores_acc_c3 = []
for train_idx, test_idx in kf_c3.split(X_feat3, Y_feat3):
    X_train, X_test = X_feat3[train_idx], X_feat3[test_idx]
    Y_train, Y_test = Y_feat3[train_idx], Y_feat3[test_idx]

    X_train_norm = scaler_c3.fit_transform(X_train)
    X_test_norm = scaler_c3.transform(X_test)

    clf_lr_c3.fit(X_train_norm, Y_train)
    scores_acc_c3.append(accuracy_score(Y_test, clf_lr_c3.predict(X_test_norm)))

Creating Epochs for PSD features...
PSD Feature shape: (244, 40)
Running CV on Feature Set 3 (PSD)...


In [ ]:
print(f"Feature Set 1 (PCA) Avg. Accuracy: {np.mean(scores_acc_c1):.4f}")
print(f"Feature Set 2 (ICA) Avg. Accuracy: {np.mean(scores_acc_c2):.4f}")
print(f"Feature Set 3 (PSD) Avg. Accuracy: {np.mean(scores_acc_c3):.4f}")

Feature Set 1 (PCA) Avg. Accuracy: 0.7459
Feature Set 2 (ICA) Avg. Accuracy: 0.7459
Feature Set 3 (PSD) Avg. Accuracy: 0.9135


## Interpretation for Part (c)

**Comparison of Results:**

* **Baseline (Part b):** 0.7459 Average Accuracy
* **Method 1 (PCA):** 0.7459 Average Accuracy
* **Method 2 (ICA):** 0.7459 Average Accuracy
* **Method 3 (PSD):** 0.9135 Average Accuracy

The **Feature Transformation** methods (PCA and ICA) showed **no improvement** over the baseline.The accuracy of 0.7459 indicates the model is still defaulting to predicting the majority class (Class 0), just as it did with the raw, high-dimensional data in Part (b).

**PCA** finds components of maximum variance. This result suggests the directions of greatest variance in the raw data are likely noise or common signal components that do not help distinguish between the "task" and "background" states.

**ICA** finds statistically independent components. While useful for artifact removal, the components themselves were not effective as features for this specific classification task.

In stark contrast, the **Feature Extraction** method (PSD Band Power) resulted in a **dramatic increase in performance** (91.35% accuracy).

* This method is based on the neurophysiological principle that different cognitive states are associated with different levels of power in specific frequency bands.
* By extracting the power in the Alpha and Beta bands, we created features that are **highly relevant** to the classification problem (Task vs. Background). The model was able to easily find a pattern in these features, whereas it found no pattern in the raw data or the PCA/ICA components.

**How to Improve Results:**

The PSD feature set is clearly the most promising. Further improvements could be achieved by:

1. **Expanding Features:** We could extract power from other brainwave bands (e.g., Theta or Gamma) to see if they add predictive information. We could also add **different types of features**, such as **Phase Synchrony**, to capture information about connectivity between channels.

2. **Optimizing Features:** We could experiment with the parameters used for feature extraction, such as the window length (`nperseg`) in the `welch` function or the length of the epochs.

3. **Improving the Model:** As required in Part (d), we can try different machine learning algorithms (e.g., KNN, SVM, Random Forest)that might find a better decision boundary in the PSD feature space. We could also tune the model's hyperparameters (e.g., the `C` parameter in `LogisticRegression`).

4. **Addressing Imbalance:** Although performance is high, we could apply the `class_weight='balanced'` parameter to the logistic regression model to ensure the class imbalance is properly handled, which might further improve the balanced accuracy or F1-score.

## Question (d) KNN Algorithm + PSD Features

In [ ]:
# Use Feature Set 3 (X_feat3, Y_feat3) from Part (c)
# Use KNN Classifier
clf_knn = KNeighborsClassifier(n_neighbors=5) # K=5
scaler_knn = StandardScaler()
kf_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_acc_d = []

# CV Loop
print("Running 5-Fold CV with KNN...")
for train_idx, test_idx in kf_knn.split(X_feat3, Y_feat3):
    X_train, X_test = X_feat3[train_idx], X_feat3[test_idx]
    Y_train, Y_test = Y_feat3[train_idx], Y_feat3[test_idx]

    # KNN is distance-based, so scaling is essential
    X_train_norm = scaler_knn.fit_transform(X_train)
    X_test_norm = scaler_knn.transform(X_test)

    clf_knn.fit(X_train_norm, Y_train) # Fit KNN
    preds_d = clf_knn.predict(X_test_norm)
    scores_acc_d.append(accuracy_score(Y_test, preds_d))

# 4. Report results
print("\n--- Part (d) Results ---")
print(f"PSD Features + KNN Avg. Accuracy: {np.mean(scores_acc_d):.4f}")

Running 5-Fold CV with KNN...

--- Part (d) Results ---
PSD Features + KNN Avg. Accuracy: 0.8974


## Interpretation for Part (d)

**Comparison of Results:**

* **Part (c) - Logistic Regression (LR) on PSD Features:** 0.9135 Average Accuracy
* **Part (d) - K-Nearest Neighbors (KNN) on PSD Features:** 0.8974 Average Accuracy

**Discussion:**
The results show that the KNN model achieved a high average accuracy of 89.74%, which is significantly better than the baseline raw data model (74.59%). However, this performance was **slightly lower** than the 91.35% accuracy achieved by the Logistic Regression model.

* **Model Differences:** Logistic Regression is a **linear model** that seeks to find an optimal linear boundary separating the classes.KNN is a non-linear, **distance-based model** that classifies new data based on the majority vote of its 'k' nearest neighbors in the feature space.
* **Interpretation:** The slightly superior performance of Logistic Regression suggests that the extracted PSD features (Alpha and Beta power) are highly **linearly separable**. This means the "task" and "background" states, when represented by these features, can be distinguished very effectively by a simple linear boundary.
* The KNN model's strong (though slightly lower) performance confirms that the PSD features create distinct, well-defined clusters in the feature space. Its performance could potentially be improved by **optimizing the 'k' parameter** (we only tested the default k=5) or by using different distance metrics.